## Config
Shared configuration for the gold notebooks. It does not process data.
Catalog and schemas come from the job parameters (bundle variables of the target).
Everything that is the same in dev and prod comes from `gold_config.yml`.

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../src"))

In [0]:
for key in ["catalog", "bronze_schema", "silver_schema", "gold_schema"]:
    dbutils.widgets.text(key, "")

In [0]:
import os
import yaml

config_path = os.path.join(os.getcwd(), "gold_config.yml")

with open(config_path, "r") as f:
    full_config = yaml.safe_load(f)

for key in ["catalog", "bronze_schema", "silver_schema", "gold_schema"]:
    value = dbutils.widgets.get(key)
    if not value:
        raise ValueError(f"Parameter '{key}' is empty: pass it from the job (base_parameters)")
    config[key] = value

In [0]:
from pyspark.sql import functions as F
from youtube_gold.scope import scope_silver

catalog = config["catalog"]
silver_schema = config["silver_schema"]
gold_schema = config["gold_schema"]

videos_silver = f"{catalog}.{silver_schema}.{config['source']['videos_silver']}"
gold = {t: f"{catalog}.{gold_schema}.{t}" for t in config["gold_tables"]}

period_start = config["trending_period"]["start"]
period_end = config["trending_period"]["end"]
reviewer_email = config["reviewer_email"]


def scoped_silver():
    """Silver rows within the business period, with a typed trending date."""
    return scope_silver(spark.table(videos_silver), period_start, period_end)